# 中转 Buffer 传输模式

本节将介绍 HIXL 的第 2 种传输模式：中转 Buffer 模式，帮助你了解该传输模式的产生背景、基本原理以及使用方法。

本节学习大纲如下：

- 中转 Buffer 模式的含义
- 中转 Buffer 模式的配置方法
- 中转 Buffer 模式的优势和限制
- 中转 Buffer 模式的性能说明


## 1. 中转 Buffer 模式的含义

中转 Buffer 模式是 HIXL 提供的一种数据传输协调机制，通过预分配的 NPU 内存池作为数据中转站，解决内存未注册场景下的传输需求。

### 1.1 产生背景

在上一节中介绍直传模式的限制时提到，当 HDK 版本低于 25.5 时，由于 RDMA 内存页管理方式的限制，注册 Host 内存存在容量上限（约 20GB），超出此限制的 Host 内存注册将失败。

为了解决 Host 注册内存溢出的问题，提出了“中转 Buffer 传输模式”的方案，该传输模式使用固定大小的已注册的中转内存作为实际传输媒介，而不需要实际注册 Host 内存，避免了直传模式下 Host 内存注册上限的问题。

### 1.2 核心原理

当源端或目标端内存未在 RDMA 通信层注册时，直接传输无法执行。中转 Buffer 模式引入第三方中转站来解决这一问题：

1. **预分配中转池**：启动时在 NPU 上分配两块大内存，分别作为客户端池和服务器池
2. **两段式传输**：数据先复制到中转 Buffer，再通过数据链路传输到对端中转 Buffer，最后复制到目标地址

下图展示了中转 Buffer 模式的数据传输流程：
<img src="./images/buffer_transfer_flow.png" alt="中转Buffer模式完整数据流转图" width="80%">

中转 Buffer 服务维护两类内存池：

| 池类型 | 索引 | 用途 |
| --- | --- | --- |
| 客户端池 | 0 | 存放本地数据副本，等待传输 |
| 服务器池 | 1 | 接收对端传输数据，作为临时中转 |

每个池由多个固定大小的 Buffer 组成，由内部内存池管理分配与回收。

### 1.3 触发条件

中转 Buffer 模式会在以下场景自动触发：

| 触发场景 | 原因说明 |
| --- | --- |
| 本地 Host 内存未注册 RDMA | 临时分配的缓冲区未经过 RDMA 注册，无法直接参与跨节点传输 |
| 远端 Host 内存未注册 RDMA | 目标地址不在 RDMA 通信范围内，需要中转缓冲 |
| 小块数据批量传输优化 | 将多个小块数据打包，减少传输启动开销 |


## 2. 中转 Buffer 模式的配置方法

### 2.1 配置参数

通过 BufferPool 参数控制中转 Buffer 池的分配：

**格式：**`"$BUFFER_NUM:$BUFFER_SIZE"`

其中：

- `BUFFER_NUM`：中转 Buffer 的数量
- `BUFFER_SIZE`：每个 Buffer 的大小，单位为MB

**默认值：**`4:8`（4 个 Buffer，每个 8MB，总池大小 32MB）

### 2.2 配置示例

```c++
std::map<AscendString, AscendString> options;

// 使用默认配置（4个8MB buffer），不设置参数时自动启用默认值
options = {};

// 自定义配置：8个16MB buffer
options["BufferPool"] = "8:16";

// 在不需要使用中转buffer进行传输的场景下，关闭中转内存池
options["BufferPool"] = "0:0";
```

### 2.3 配置约束

1. **禁用配置**：设置`"0:0"`将完全关闭中转池，未注册内存的传输请求会失败
2. **内存占用**：总池大小为`BUFFER_NUM × BUFFER_SIZE × 2`（两池），需确保 NPU 内存充足


## 3. 中转 Buffer 模式的优势和限制

### 3.1 小块数据传输优化

对于小于 256KB 的数据块，中转 Buffer 模式可以提供性能优势：

**问题**：每次 RDMA 传输都有启动开销（建立连接、准备描述符等），小块数据的传输时间可能低于启动开销，导致效率低下。

**解决方案**：将多个小块数据打包到单个中转 Buffer，一次传输完成批量搬迁。

### 3.2 使用限制

**1. 仅支持同步传输**

中转 Buffer 服务仅处理同步传输接口，异步接口不经过 Buffer 服务。这意味着使用中转 Buffer 时，调用线程会被阻塞直到传输完成。

**2. 额外中转开销**

每次传输增加两段本地复制操作：

<img src="./images/buffer_transfer_overhead.png" alt="中转Buffer开销示意" width="60%">

- **Client Copy**：本地数据复制到中转 Buffer（Host→Device 或 Device→Device）
- **Server Copy**：从中转 Buffer 复制到目标地址（Device→Host 或 Device→Device）

直传模式就像"门到门"快递，一步到位；中转 Buffer 模式则需要"寄件人→中转站→收件人"三步走，每一步都有时间和资源消耗。


## 4. 中转 Buffer 模式的性能说明

### 4.1 实测性能

下表只展示 A2/A3 场景下中转 Buffer 模式的实测 WRITE 写带宽，详细实测性能请查看 [A2 benchmark数据](https://gitcode.com/cann/hixl/blob/master/benchmarks/A2_benchmark_performance.md) 和 [A3 benchmark数据](https://gitcode.com/cann/hixl/blob/master/benchmarks/A3_benchmark_performance.md)。

- A2 场景的性能：

| 链路类型 | 机内实测带宽(GB/s) | 跨机实测带宽(GB/s) |
| --- | --- | --- |
| HCCS D2rD | 19 | 22 |
| RoCE D2rD | 23 | 23 |
| HCCS H2rD | 14 | 14 |
| RoCE H2rD | 16 | 14 |
| HCCS D2rH | 11 | 15 |
| RoCE D2rH | 18 | 15 |
| HCCS H2rH | 11 | 12 |
| RoCE H2rH | 16 | 14 |

- A3 场景的性能：

| 链路类型 | 机内实测带宽(GB/s) | 跨机实测带宽(GB/s) |
----------|----------|----------|
| HCCS D2rD | 115 | 110 |
| RoCE D2rD | 22 | 33 |
| HCCS H2rD | 34 | 13 |
| RoCE H2rD | 14 | 22 |
| HCCS D2rH | 26 | 27 |
| RoCE D2rH | 15 | 15 |
| HCCS H2rH | 26 | 26 |
| RoCE H2rH | 15 | 16 |

### 4.1 与直传模式对比

| 对比项 | 直传模式 | 中转 Buffer 模式 |
| --- | --- | --- |
| **传输路径** | 源端内存 → 链路 → 目标端内存 | 源端 → 中转 Buffer → 链路 → 中转 Buffer → 目标端 |
| **内存要求** | 内存需 RDMA 注册 | 支持未注册内存 |
| **传输接口** | 同步+异步双模式 | 仅支持同步 |
| **操作次数** | 1 次 RDMA 传输 | 3 次操作叠加 |
| **延迟** | 低 | 高 |

> 中转 Buffer 模式是一种兼容性方案，适用于无法使用更优方案的临时场景。生产环境建议使用 HDK 大于 25.5 的版本，优先使用直传模式或后续章节的 Fabricmem 模式以获得更优的性能。


## 课后练习

本节介绍了中转 Buffer 模式的原理、配置方法和使用限制，请根据学习内容完成以下题目进行自测。

1. （判断题）中转 Buffer 模式通过预分配的 NPU 内存池作为数据中转站，解决内存未注册场景下的传输需求。

2. （判断题）中转 Buffer 模式推荐在生产环境使用，因为它可以自动处理未注册内存的传输。

3. （单选题）BufferPool 参数的格式为"BUFFER_NUM : BUFFER_SIZE"，其中 BUFFER_SIZE 的单位是什么？
    A. Byte
    B. KB
    C. MB
    D. GB

4. （单选题）关于中转 Buffer 模式与直传模式的对比，以下说法正确的是？
    A. 中转 Buffer 模式性能优于直传模式
    B. 两者传输路径相同，都是直接传输
    C. 直传模式支持同步和异步双模式，中转 Buffer 仅支持同步
    D. 中转 Buffer 模式是 HIXL 的默认传输模式

5. （多选题）中转 Buffer 模式有哪些使用限制？
    A. 仅支持同步传输
    B. 每次传输增加两段本地复制操作
    C. 大数据需要多次分片传输
    D. 支持异步传输接口

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/03.03_answer.txt